# Fine-Tune LoRA for JEMS Agent with Qwen2.5-3B-Instruct

This notebook trains a lightweight **LoRA (Low-Rank Adaptation)** adapter on top of **`Qwen/Qwen2.5-3B-Instruct`** (the base model used in the JEMS project).

### Why LoRA for JEMS?
- **VRAM Efficient**: LoRA adds small trainable matrices (~30MB–80MB) while keeping the 3B base model frozen.
- **Multi-Task Serving**: Once trained, your single vLLM server can serve multiple LoRAs simultaneously (e.g. `coding-lora`, `roadmap-lora`) alongside the base `vllm` model with virtually zero extra GPU RAM.
- **Fast & Free**: Can be trained in 10–20 minutes on a free Google Colab T4 GPU (or locally if you have CUDA enabled).

## 1. Install Dependencies
We use **Unsloth** (or Hugging Face PEFT/TRL), which speeds up training by 2x-5x and reduces memory by 70%.

In [ ]:
# Install unsloth, trl, peft and transformers
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers" "trl<0.9.0" peft accelerate bitsandbytes datasets

## 2. Load Base Model & Setup LoRA
We load `Qwen/Qwen2.5-3B-Instruct` in 4-bit quantization and attach LoRA target modules.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048   # Context length
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,             # None for auto detection (Float16 on T4, Bfloat16 on Ampere)
    load_in_4bit=True,      # Uses 4-bit QLoRA to fit comfortably in <6GB VRAM
)

# Attach LoRA adapter matrices
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                   # LoRA rank (16 or 32 is ideal for task specialization)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha=16,
    lora_dropout=0,         # 0 is optimized by Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("LoRA Model configured successfully!")

## 3. Prepare Dataset (ChatML Format)
Qwen2.5 uses the **ChatML** prompt format (`<|im_start|>system...<|im_end|><|im_start|>user...<|im_end|><|im_start|>assistant...<|im_end|>`).

Below is an example training dataset for JEMS Agent tasks (e.g. generating coding problems or detailed roadmaps in structured JSON). Replace or expand this list with your own domain examples.

In [ ]:
from datasets import Dataset

# Sample training data for specialized agent output
training_data = [
    {
        "messages": [
            {"role": "system", "content": "You are a coding exam generator. Reply with valid JSON matching the schema."},
            {"role": "user", "content": "Create a coding problem for Core Java on the topic of Collections (HashMap)."},
            {"role": "assistant", "content": '{"title": "Word Frequency Counter", "difficulty": "medium", "language": "java", "description": "Given a sentence of space-separated words, count the frequency of each unique word using a HashMap and print them sorted alphabetically.", "starter_code": "import java.util.*;\n\npublic class Main {\n    public static void main(String[] args) {\n        Scanner sc = new Scanner(System.in);\n        // TODO: implement\n    }\n}", "test_cases": [{"input": "apple banana apple orange", "expected_output": "apple: 2\\nbanana: 1\\norange: 1"}]}'}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a coding exam generator. Reply with valid JSON matching the schema."},
            {"role": "user", "content": "Create an easy Python problem on String reversal."},
            {"role": "assistant", "content": '{"title": "Reverse Words", "difficulty": "easy", "language": "python", "description": "Read a line of text from standard input and print each word reversed while keeping their order.", "starter_code": "import sys\n\ndef main():\n    # write solution here\n    pass\n\nif __name__ == \"__main__\":\n    main()", "test_cases": [{"input": "hello world", "expected_output": "olleh dlrow"}]}'}
        ]
    },
    {
        "messages": [
            {"role": "system", "content": "You are a senior curriculum designer. Reply with ONE JSON object."},
            {"role": "user", "content": "Write the syllabus outline for Docker Containerization prerequisite step."},
            {"role": "assistant", "content": '{"weeks": 2, "summary": "Master container basics, Dockerfiles, and multi-container orchestration with Docker Compose.", "why_learn": "Containers ensure reproducible environments and form the foundation for modern cloud deployments.", "used_in": ["Packaging microservices into lightweight images", "Running local development databases with Docker Compose", "Deploying workloads to Kubernetes clusters"], "topics": ["Container Fundamentals: namespaces and cgroups", "Dockerfile syntax: FROM, RUN, COPY, CMD", "Container Networking and Port Forwarding", "Persistent Storage with Docker Volumes", "Docker Compose for Multi-Container Apps", "Image Optimization and Multi-Stage Builds"], "project": "Dockerize a fullstack web application with PostgreSQL", "resources": ["Docker Official Docs", "Play with Docker Labs"]}'}
        ]
    }
]

# Format conversations using Qwen's official chat template
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

dataset = Dataset.from_list(training_data)
dataset = dataset.map(formatting_prompts_func, batched=True)
print("Sample Formatted Input:\n", dataset["text"][0][:400], "...")

## 4. Train with SFTTrainer
We configure `SFTTrainer` with standard hyperparameters.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,               # Set to e.g. 100-300 for real datasets
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="lora_checkpoints",
    ),
)

# Start Fine-Tuning
trainer_stats = trainer.train()
print("Training Complete!", trainer_stats)

## 5. Test Inference with LoRA
Test that the fine-tuned LoRA adapter generates proper structured output.

In [ ]:
FastLanguageModel.for_inference(model)  # Enables 2x faster native inference

test_messages = [
    {"role": "system", "content": "You are a coding exam generator. Reply with valid JSON matching the schema."},
    {"role": "user", "content": "Create a coding problem for Python on List comprehensions."}
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True)
print(tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0])

## 6. Export LoRA Adapter for vLLM
Save the LoRA weights (`adapter_config.json` and `adapter_model.safetensors`).

These files will be placed into `Agent/loras/<adapter_name>` so vLLM can load them.

In [ ]:
LORA_OUTPUT_DIR = "coding-lora"

# Save LoRA adapter locally
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)
print(f"LoRA adapter files successfully saved in '{LORA_OUTPUT_DIR}'!")

# Optional: Push directly to Hugging Face Hub if you have a HF token
# model.push_to_hub("your-username/qwen2.5-3b-coding-lora", token="hf_...")

## 7. How to Load in JEMS vLLM Docker

1. **Copy the adapter folder** into your project:
   ```text
   JEMS/
     Agent/
       loras/
         coding-lora/
           adapter_config.json
           adapter_model.safetensors
   ```

2. **Enable LoRA in `Agent/docker-compose.yml`** under the `vllm` service command:
   ```yaml
   command:
     - --model=${VLLM_MODEL:-Qwen/Qwen2.5-3B-Instruct-AWQ}
     - --served-model-name=vllm
     - --enable-lora
     - --max-loras=2
     - --max-lora-rank=16
     - --lora-modules=coding-lora=/root/loras/coding-lora
   volumes:
     - hf_cache:/root/.cache/huggingface
     - ./loras:/root/loras:ro
   ```

3. **Call it in Agent API requests**:
   Use `model: "coding-lora"` to trigger this LoRA adapter, or `model: "vllm"` for the base model.